In [5]:
import pandas as pd, unicodedata

ARQ    = 'Clientes'   # único campo que muda de tabela para tabela
APAGAR = []               # opcional: colunas com dado que você quer remover (ex.: ['id_registro'])

sem_acento = lambda t: unicodedata.normalize('NFKD', str(t)).encode('ASCII', 'ignore').decode('ASCII')

df = pd.read_csv(f'../base_csv/original/{ARQ}.csv', encoding='latin1', sep=None, engine='python', dtype=str)

# cabeçalho: minúsculo, sem acento, espaço vira _
df.columns = [sem_acento(c).strip().lower().replace(' ', '_') for c in df.columns]
df = df.drop(columns=APAGAR, errors='ignore')

# "R$ -", "-" e vazio viram nulo; remove linhas e colunas 100% vazias
limpa = lambda s: s.str.strip().mask(s.str.strip().str.fullmatch(r'(R\$)?\s*-?\s*', na=True))
df = df.apply(limpa).dropna(how='all').dropna(axis=1, how='all')

for c in df.columns:
    v = df[c].dropna()
    if v.str.fullmatch(r'\d{2}/\d{2}/\d{4}').all():
        df[c] = pd.to_datetime(df[c], dayfirst=True).dt.strftime('%Y-%m-%d')
        print(c, '-> data')
    elif v.str.fullmatch(r'(R\$\s*)?-?[\d.]*\d,\d{1,2}').all():
        df[c] = pd.to_numeric(df[c].str.replace(r'[R$\s.]', '', regex=True).str.replace(',', '.'))
        print(c, '-> decimal')
    else:
        df[c] = df[c].map(lambda t: sem_acento(t) if pd.notna(t) else t)

# Supabase: vírgula, UTF-8
df.to_csv(f'../base_csv/tratado/{ARQ}_tratado.csv', index=False, sep=';', encoding='utf-8-sig')

print(df.columns.tolist(), df.isna().sum().sum(), "vazios")

registro -> data
['id_cliente', 'cliente', 'cnpj', 'registro'] 0 vazios
